# **Actividad 2: Modelos de predicción y clasificación** 
En este notebook se construye el flujo de trabajo completo para la segunda parte del proyecto modelos de predicción y clasificación del dataset llamado "California housing" o en español "Viviendas de California".

In [14]:
# Importaciones 
import pandas as pd
import numpy as np
from datasets import load_dataset
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## **0. Construcción automática del DataFrame**

**Dataset:** California Housing, basado en el censo de California de 1990.  
**Identificador:** `BrianSuToronto/california_housing`  
**Enlace:** https://huggingface.co/datasets/BrianSuToronto/california_housing  
**Split utilizado:** `train`  
**Contenido:** Cada fila describe una zona (bloque del censo) con sus características demográficas, de vivienda y de ubicación.  
**Columnas relevantes (ya renombradas):** `Ingreso_mediano`, `Edad_vivienda`, `Promedio_habitaciones`, `Promedio_dormitorio_hogar`, `Poblacion`, `Promedio_ocupacion`, `Latitud`, `Longitud` y `Valor_mediano_vivienda`.    
**Problema de regresión:** Predecir `Valor_mediano_vivienda` (numérica continua).  
**Problema de clasificación:** Predecir `Categoria_valor` (Bajo, Medio, Alto), construida a partir de `Valor_mediano_vivienda`.

### **Creación y carga del dataset** 

In [15]:
DATASET = "BrianSuToronto/california_housing"   
SPLIT = "train"

ds = load_dataset(DATASET)
print("Splits disponibles:", list(ds.keys()))
df = ds[SPLIT].to_pandas()

Splits disponibles: ['train']


### **Renombrado de columnas**

In [16]:
nombres_columnas = {
    "MedInc": "Ingreso_mediano",
    "HouseAge": "Edad_vivienda",
    "AveRooms": "Promedio_habitaciones",
    "AveBedrms": "Promedio_dormitorio_hogar",
    "Population": "Poblacion",
    "AveOccup": "Promedio_ocupacion",
    "Latitude": "Latitud",
    "Longitude": "Longitud",
    "MedHouseVal": "Valor_mediano_vivienda",
}

df_transformado = df.rename(columns=nombres_columnas)
print(df_transformado.shape)
display(df_transformado.head(3))

(20640, 9)


,Ingreso_mediano,Edad_vivienda,Promedio_habitaciones,Promedio_dormitorio_hogar,Poblacion,Promedio_ocupacion,Latitud,Longitud,Valor_mediano_vivienda
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521


### **Mostramos contenido del DataFrame**

In [17]:
# Mostramos el DataFrame
print("DataFrame de 'California_Housing'")
print("\n- Tamaño (filas, columnas):", df_transformado.shape)
print("- Columnas:", df_transformado.columns.tolist())
print("- Tipos de datos:")
print(df_transformado.dtypes)
print("\nContenido del DataFrame:")
display(df_transformado.head(3))

DataFrame de 'California_Housing'

- Tamaño (filas, columnas): (20640, 9)
- Columnas: ['Ingreso_mediano', 'Edad_vivienda', 'Promedio_habitaciones', 'Promedio_dormitorio_hogar', 'Poblacion', 'Promedio_ocupacion', 'Latitud', 'Longitud', 'Valor_mediano_vivienda']
- Tipos de datos:
Ingreso_mediano              float64
Edad_vivienda                float64
Promedio_habitaciones        float64
Promedio_dormitorio_hogar    float64
Poblacion                    float64
Promedio_ocupacion           float64
Latitud                      float64
Longitud                     float64
Valor_mediano_vivienda       float64
dtype: object

Contenido del DataFrame:


,Ingreso_mediano,Edad_vivienda,Promedio_habitaciones,Promedio_dormitorio_hogar,Poblacion,Promedio_ocupacion,Latitud,Longitud,Valor_mediano_vivienda
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521


### **Definición de las variables objetivo (regresión y clasificación)**

In [18]:
# Regla: tres categorías con la misma cantidad de zonas (terciles)
df_modelo = df_transformado.assign(
    Categoria_valor=pd.qcut(
        df_transformado["Valor_mediano_vivienda"],
        q=3,
        labels=["Bajo", "Medio", "Alto"],
    )
)

print(df_modelo["Categoria_valor"].value_counts())
print(df_modelo.groupby("Categoria_valor", observed=False)["Valor_mediano_vivienda"].agg(["min", "max"]))

Categoria_valor
Bajo     6884
Alto     6880
Medio    6876
Name: count, dtype: int64
                     min      max
Categoria_valor                  
Bajo             0.14999  1.41400
Medio            1.41500  2.30300
Alto             2.30400  5.00001


**Regla de clasificación:** Se creó una variable categórica a partir de la variable numérica `Valor_mediano_vivienda`, dividiéndola en terciles con `pd.qcut`, de modo que cada categoría (Bajo, Medio, Alto) tiene aproximadamente la misma cantidad de zonas. Se eligió esta regla porque deja clases balanceadas, lo que evita que el modelo aprenda a predecir siempre la clase mayoritaria. La variable `Valor_mediano_vivienda` no se incluye entre las predictoras de clasificación, porque revelaría directamente la respuesta.

## **1. Evidencia de preprocesamiento**

### **Estado inicial**

In [19]:
# ANTES este es el estado inicial 
print("Tamaño:", df_modelo.shape)
print("\nTipos:\n", df_modelo.dtypes)
print("\nNulos por columna:\n", df_modelo.isna().sum())
print("\nDuplicados:", df_modelo.duplicated().sum())

display(df_modelo.describe().T)

# Asimetría y valores atípicos (regla del IQR) de las predictoras
predictoras = [c for c in df_modelo.columns if c not in ["Valor_mediano_vivienda", "Categoria_valor"]]
asimetria_antes = df_modelo[predictoras].skew().round(2)
print("\nAsimetría antes:\n", asimetria_antes)

q1, q3 = df_modelo[predictoras].quantile(0.25), df_modelo[predictoras].quantile(0.75)
iqr = q3 - q1
atipicos = ((df_modelo[predictoras] < q1 - 1.5 * iqr) | (df_modelo[predictoras] > q3 + 1.5 * iqr)).sum()
print("\nValores atípicos por columna:\n", atipicos)

Tamaño: (20640, 10)

Tipos:
 Ingreso_mediano               float64
Edad_vivienda                 float64
Promedio_habitaciones         float64
Promedio_dormitorio_hogar     float64
Poblacion                     float64
Promedio_ocupacion            float64
Latitud                       float64
Longitud                      float64
Valor_mediano_vivienda        float64
Categoria_valor              category
dtype: object

Nulos por columna:
 Ingreso_mediano              0
Edad_vivienda                0
Promedio_habitaciones        0
Promedio_dormitorio_hogar    0
Poblacion                    0
Promedio_ocupacion           0
Latitud                      0
Longitud                     0
Valor_mediano_vivienda       0
Categoria_valor              0
dtype: int64

Duplicados: 0


,count,mean,std,min,25%,50%,75%,max
Ingreso_mediano,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
Edad_vivienda,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
Promedio_habitaciones,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
Promedio_dormitorio_hogar,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Poblacion,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
Promedio_ocupacion,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitud,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitud,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000
Valor_mediano_vivienda,20640.0,2.068558,1.153956,0.149990,1.196000,1.797000,2.647250,5.000010



Asimetría antes:
 Ingreso_mediano               1.65
Edad_vivienda                 0.06
Promedio_habitaciones        20.70
Promedio_dormitorio_hogar    31.32
Poblacion                     4.94
Promedio_ocupacion           97.64
Latitud                       0.47
Longitud                     -0.30
dtype: float64

Valores atípicos por columna:
 Ingreso_mediano               681
Edad_vivienda                   0
Promedio_habitaciones         511
Promedio_dormitorio_hogar    1424
Poblacion                    1196
Promedio_ocupacion            711
Latitud                         0
Longitud                        0
dtype: int64


El dataset contiene 9 variables numéricas de tipo float y solo una categorica que fue la agregada por nosotros anteriormente, no contiene datos  nulos, ni duplicados, las columnas que contienen más valores atípicos son Promedio_dormitorio_hogar y Poblacion , siguiendoles Promedio_ocupacion e Ingreso_mediano.

### **Operaciones realizadas (No aprenden de los datos)**

#### **Duplicados**

In [20]:
# Verificación de duplicados
# No hay duplicados solo se copia el dataframe
df_prep = df_modelo.copy()
print("Duplicados encontrados:", df_modelo.duplicated().sum())
print("Filas:", len(df_prep))

Duplicados encontrados: 0
Filas: 20640


#### **Enriquecimiento de variables**

In [21]:
# Enriquecimiento de variables 

# Evidencia que justifica la variable nueva (se calcula antes del logaritmo)
print("Correlación entre habitaciones y dormitorios:")
print(df_prep[["Promedio_habitaciones", "Promedio_dormitorio_hogar"]].corr().round(3))

# Creación de la variable: habitaciones por cada dormitorio
df_prep = df_prep.assign(
    Habitaciones_por_dormitorio=df_prep["Promedio_habitaciones"] / df_prep["Promedio_dormitorio_hogar"]
)

# Validación: no debe haber infinitos ni nulos
invalidos = df_prep["Habitaciones_por_dormitorio"].replace([np.inf, -np.inf], np.nan).isna().sum()
print("\nValores infinitos o nulos:", invalidos)

# Muestra y estadísticas de la variable nueva
display(df_prep[["Promedio_habitaciones", "Promedio_dormitorio_hogar", "Habitaciones_por_dormitorio"]].head())
display(df_prep["Habitaciones_por_dormitorio"].describe())

# Se agrega a las predictoras (solo si no está) y se recalcula la asimetría "antes"
if "Habitaciones_por_dormitorio" not in predictoras:
    predictoras.append("Habitaciones_por_dormitorio")

asimetria_antes = df_prep[predictoras].skew().round(2)
print("\nNúmero de predictoras:", len(predictoras))

Correlación entre habitaciones y dormitorios:
                           Promedio_habitaciones  Promedio_dormitorio_hogar
Promedio_habitaciones                      1.000                      0.848
Promedio_dormitorio_hogar                  0.848                      1.000

Valores infinitos o nulos: 0


,Promedio_habitaciones,Promedio_dormitorio_hogar,Habitaciones_por_dormitorio
0,6.984127,1.023810,6.821705
1,6.238137,0.971880,6.418626
2,8.288136,1.073446,7.721053
3,5.817352,1.073059,5.421277
4,6.281853,1.081081,5.810714


count    20640.000000
mean         4.984262
std          1.171836
min          1.000000
25%          4.169542
50%          4.921728
75%          5.700424
max         10.000000
Name: Habitaciones_por_dormitorio, dtype: float64


Número de predictoras: 9


Las columnas `Promedio_habitaciones` y `Promedio_dormitorio_hogar` tienen una correlación de **0.848** (calculada con `corr()`): están relacionadas, pero cada una aporta información propia. La variable `Habitaciones_por_dormitorio` resume cuántas habitaciones hay por cada dormitorio, una relación que un modelo lineal no puede formar por sí solo a partir de las dos columnas por separado.

#### **Transformaciones logarítmicas**

In [22]:
# Transformación logarítmica de variables muy sesgadas
df_prep = df_prep.assign(
    Promedio_ocupacion=np.log1p(df_prep["Promedio_ocupacion"]),
    Promedio_dormitorio_hogar=np.log1p(df_prep["Promedio_dormitorio_hogar"]),
    Promedio_habitaciones=np.log1p(df_prep["Promedio_habitaciones"]),
    Poblacion=np.log1p(df_prep["Poblacion"])
)

asimetria_despues = df_prep[predictoras].skew().round(2)
display(pd.DataFrame({"antes": asimetria_antes, "despues": asimetria_despues}))

,antes,despues
Ingreso_mediano,1.65,1.65
Edad_vivienda,0.06,0.06
Promedio_habitaciones,20.70,1.39
Promedio_dormitorio_hogar,31.32,8.99
Poblacion,4.94,-1.04
Promedio_ocupacion,97.64,3.88
Latitud,0.47,0.47
Longitud,-0.30,-0.30
Habitaciones_por_dormitorio,0.34,0.34


La transformación logarítmica redujo de forma importante la asimetría de las cuatro columnas. `Promedio_habitaciones` quedó casi simétrica (1.39), mientras que `Promedio_ocupacion` (3.88) y `Promedio_dormitorio_hogar` (8.99) y `Poblacion` (-1.04) y mejoraron mucho, pero siguen sesgadas por valores atípicos extremos. `Habitaciones_por_dormitorio` (0.34), `Edad_vivienda`, `Latitud` y `Longitud` ya eran casi simétricas y no se transformaron.

### **Separamos entrenamiento y prueba**

In [23]:
train_df, test_df = train_test_split(
    df_prep, test_size=0.2, random_state=42, stratify=df_prep["Categoria_valor"]
)

X_train, X_test = train_df[predictoras], test_df[predictoras]
y_train_reg, y_test_reg = train_df["Valor_mediano_vivienda"], test_df["Valor_mediano_vivienda"]
y_train_clf, y_test_clf = train_df["Categoria_valor"], test_df["Categoria_valor"]

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print(y_train_clf.value_counts(normalize=True).round(3))
print(y_test_clf.value_counts(normalize=True).round(3))

Entrenamiento: (16512, 9) | Prueba: (4128, 9)
Categoria_valor
Bajo     0.334
Alto     0.333
Medio    0.333
Name: proportion, dtype: float64
Categoria_valor
Bajo     0.334
Alto     0.333
Medio    0.333
Name: proportion, dtype: float64


### **Operaciones realizadas (Aprenden con entrenamiento)**  

#### **Escalado**

In [24]:
# Escalado
scaler = StandardScaler()
X_train_esc = pd.DataFrame(
    scaler.fit_transform(X_train), columns=predictoras, index=X_train.index
)
X_test_esc = pd.DataFrame(
    scaler.transform(X_test), columns=predictoras, index=X_test.index
)

### **Comparación de antes y después del escalado**

In [25]:
comparacion = pd.DataFrame({
    "media_antes": X_train.mean().round(3),
    "std_antes": X_train.std().round(3),
    "media_despues": X_train_esc.mean().round(3),
    "std_despues": X_train_esc.std().round(3),
})
display(comparacion)

,media_antes,std_antes,media_despues,std_despues
Ingreso_mediano,3.876,1.910,0.0,1.0
Edad_vivienda,28.642,12.595,0.0,1.0
Promedio_habitaciones,1.830,0.233,-0.0,1.0
Promedio_dormitorio_hogar,0.732,0.107,-0.0,1.0
Poblacion,7.027,0.735,0.0,1.0
Promedio_ocupacion,1.352,0.205,0.0,1.0
Latitud,35.644,2.139,-0.0,1.0
Longitud,-119.579,2.007,-0.0,1.0
Habitaciones_por_dormitorio,4.989,1.176,-0.0,1.0


### **Justificación de las decisiones**

| Decisión | Problema que resuelve | Efecto en los datos |
|---|---|---|
| Carga automática con `load_dataset` | La actividad exige que los datos se obtengan por código, sin descargas manuales | El DataFrame se construye desde Hugging Face en cada ejecución, lo que hace el análisis reproducible |
| Renombrar las columnas al español | Los nombres originales en inglés son abreviados (`MedInc`, `AveRooms`) y poco claros | Las 9 columnas conservan sus datos y ahora tienen nombres descriptivos |
| Crear `Categoria_valor` con terciles (`pd.qcut`) | La clasificación necesita una variable objetivo categórica y el dataset solo trae una numérica | Se obtienen tres clases (Bajo, Medio, Alto) con aproximadamente la misma cantidad de zonas, lo que evita que el modelo aprenda a predecir siempre la clase mayoritaria |
| Excluir `Valor_mediano_vivienda` y `Categoria_valor` de las predictoras | Si la variable que generó la etiqueta entra como predictora, el modelo "ve" la respuesta (fuga de información) | Las predictoras son solo las 8 características de la zona, más la variable creada en el enriquecimiento |
| Verificar nulos y duplicados sin aplicar cambios | Los nulos y las filas repetidas pueden sesgar el entrenamiento | `isna().sum()` y `duplicated().sum()` dieron 0, por lo que no se aplicó imputación ni eliminación de duplicados |
| Enriquecimiento: `Habitaciones_por_dormitorio` | `Promedio_habitaciones` y `Promedio_dormitorio_hogar` están moderadamente correlacionadas (0.518) y ninguna expresa por sí sola la proporción entre ambas | Se agregó una variable que resume cuántas habitaciones hay por cada dormitorio (de 8 a 9 predictoras). Se calculó antes del logaritmo para conservar su significado original |
| Transformación logarítmica (`Promedio_ocupacion`, `Promedio_dormitorio_hogar`, `Promedio_habitaciones`, `Poblacion`) | Asimetría fuerte y valores atípicos extremos, que pueden dominar el ajuste del modelo | La asimetría bajó de 97.64 a 3.88 en `Promedio_ocupacion`, de 31.32 a 8.99 en `Promedio_dormitorio_hogar`, de 20.70 a 1.39 en `Promedio_habitaciones` y de 4.94 a -1.04 en `Poblacion` |
| División entrenamiento/prueba (80/20, estratificada, `random_state=42`) | Un modelo evaluado con los mismos datos con que se entrenó da una medida demasiado optimista | Hay un conjunto para entrenar y otro que el modelo no vio. La estratificación mantiene proporciones de clases casi iguales en ambos y la semilla fija permite repetir el resultado |
| Escalado con `StandardScaler`, ajustado solo con entrenamiento | Las variables tienen escalas muy distintas (por ejemplo, `Poblacion` frente a `Latitud`), lo que perjudica a modelos como la regresión logística | En entrenamiento la media queda en 0 y la desviación en 1. El escalador se ajustó únicamente con entrenamiento y luego se aplicó a prueba, para que ningún dato de prueba influya en el preprocesamiento |

**Operaciones que no se aplicaron**
- **Imputación de nulos:** el dataset no tiene valores nulos, por lo que no era necesaria.
- **Eliminación de duplicados:** no se encontraron filas repetidas.
- **Codificación de categorías:** todas las predictoras son numéricas. `Categoria_valor` se usa solo como objetivo de clasificación.
- **Escalado del objetivo de regresión:** `Valor_mediano_vivienda` se mantiene en su escala original para que los errores del modelo se interpreten en las mismas unidades que el valor de las viviendas.
- **Transformación de `Ingreso_mediano`, `Habitaciones_por_dormitorio`, `Edad_vivienda`, `Latitud` y `Longitud`:** su asimetría es baja (entre -0.30 y 1.65), por lo que no la necesitan.

**Limitaciones**
- Los terciles de `Categoria_valor` se calcularon con todo el dataset antes de dividir. Se aceptó porque definen la etiqueta y no una característica de las zonas.
- Tras el logaritmo, `Promedio_ocupacion` (3.88) y `Promedio_dormitorio_hogar` (8.99) siguen con asimetría alta por valores atípicos extremos, aunque su influencia es mucho menor que antes.

### **Resultado final**
Datos listos para modelar

In [26]:
print("Entrenamiento:", X_train_esc.shape, "| Prueba:", X_test_esc.shape)
display(X_train_esc.head())

Entrenamiento: (16512, 9) | Prueba: (4128, 9)


,Ingreso_mediano,Edad_vivienda,Promedio_habitaciones,Promedio_dormitorio_hogar,Poblacion,Promedio_ocupacion,Latitud,Longitud,Habitaciones_por_dormitorio
8411,-0.538992,0.504828,-1.439851,-0.202019,0.702868,1.719484,-0.796794,0.612544,-1.406176
5456,-0.608042,0.743028,-1.606736,-0.195030,-0.318450,-0.667431,-0.768740,0.552747,-1.549868
3490,0.899015,-0.447971,0.420021,-0.058553,0.133202,1.623927,-0.637823,0.557730,0.440092
10031,-0.572968,0.504828,0.174768,-0.070134,0.064709,-0.537750,1.676607,-0.752804,0.145715
9566,-0.738394,0.425428,0.156862,-0.352934,-0.298755,0.008019,0.797591,-0.508636,0.391401
